# Match Christian's neuron count on zplane02

**Goal:** reuse Christian's exact `data.bin` and `ops.npy` from `D:/jeff/cjennings/zplane02_tp00001-08440/` and re-run suite2p detection. If we can't reproduce his ROI count (1506 total / ~791 accepted) given identical inputs, the gap is traceable to a specific code-path or library-version difference rather than data preprocessing.

**Why bypass `lsp.pipeline`:** the LBM fork's `run_lsp.py:2334` overrides `ops['fs']` from TIFF metadata when `fs in (None, 10.0)`, which clobbers Christian's `fs=10` even when set explicitly. Bin size for detection is `round(tau*fs)` so this directly affects what cellpose sees. We call `suite2p.pipeline` directly (the flat-ops `run_s2p.pipeline` exposed at the package root) to avoid that path.


In [1]:
from pathlib import Path
import copy
import logging
import numpy as np
import torch

# Make suite2p / cellpose progress visible so the run is auditable.
logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s", force=True)
for name in ("suite2p", "cellpose"):
    logging.getLogger(name).setLevel(logging.INFO)

SRC = Path("D:/jeff/cjennings/zplane02_tp00001-08440")
OUT = Path("D:/jeff/cjennings/zplane02_replication")
OUT.mkdir(parents=True, exist_ok=True)

assert (SRC / "data.bin").exists(), f"missing data.bin at {SRC}"
assert (SRC / "ops.npy").exists(), f"missing ops.npy at {SRC}"
print(f"src: {SRC}")
print(f"out: {OUT}")

src: D:\jeff\cjennings\zplane02_tp00001-08440
out: D:\jeff\cjennings\zplane02_replication


In [2]:
# Load Christian's reference and print the targets we want to match.
ref_ops = np.load(SRC / "ops.npy", allow_pickle=True).item()
ref_stat = np.load(SRC / "stat.npy", allow_pickle=True)
ref_iscell = np.load(SRC / "iscell.npy")

print("=== Christian's reference ===")
print(f"  ROIs total   : {len(ref_stat)}")
print(f"  ROIs accepted: {int(ref_iscell[:, 0].sum())}")
print()
for k in ("fs", "tau", "diameter", "diameter_user", "anatomical_only",
          "threshold_scaling", "cellprob_threshold", "flow_threshold",
          "spatial_hp_cp", "spatial_scale", "high_pass", "max_overlap",
          "pretrained_model", "sparse_mode", "soma_crop", "smooth_sigma",
          "nbinned", "nframes", "Ly", "Lx"):
    v = ref_ops.get(k, "<missing>")
    if isinstance(v, np.ndarray):
        v = v.tolist() if v.size <= 4 else f"<arr {v.shape}>"
    print(f"  {k:22s} = {v}")

bin_size = int(max(1, ref_ops["nframes"] // ref_ops["nbinned"],
                   round(ref_ops["tau"] * ref_ops["fs"])))
print(f"\n  expected bin_size (round(tau*fs)) = {bin_size}")

=== Christian's reference ===
  ROIs total   : 1506
  ROIs accepted: 791

  fs                     = 10.0
  tau                    = 1.3
  diameter               = 6.076507568359375
  diameter_user          = 2
  anatomical_only        = 4
  threshold_scaling      = 1.0
  cellprob_threshold     = -6
  flow_threshold         = 0
  spatial_hp_cp          = 3
  spatial_scale          = 1
  high_pass              = 100
  max_overlap            = 1.0
  pretrained_model       = cpsam
  sparse_mode            = True
  soma_crop              = True
  smooth_sigma           = 1.15
  nbinned                = 5000
  nframes                = 8440
  Ly                     = 1002
  Lx                     = 725

  expected bin_size (round(tau*fs)) = 13


In [3]:
# Build a fresh ops dict pointing at the OUT dir but reading Christian's data.bin.
# We keep all his params (fs=10, diameter=6.08, threshold_scaling=1.0, ...) and
# clear only the detection outputs we want recomputed. Registration intermediates
# (meanImg, meanImgE, yrange, xrange, xoff, ...) are kept because do_registration=0
# means suite2p will read them from ops rather than recompute.
ops = copy.deepcopy(ref_ops)
ops["save_path"] = str(OUT)
ops["ops_path"] = str(OUT / "ops.npy")
ops["reg_file"] = str(SRC / "data.bin")           # ← reuse Christian's binary verbatim
ops["raw_file"] = str(SRC / "data_raw.bin")
ops["do_registration"] = 0
ops["roidetect"] = 1

# clear stale detection outputs so they get recomputed
for k in ("stat", "F", "Fneu", "spks", "iscell", "redcell",
         "detect_outputs", "nrois"):
    ops.pop(k, None)

# upstream pipeline saves files via ops['save_path'] - persist before running
np.save(OUT / "ops.npy", ops, allow_pickle=True)
print(f"  ops['fs']        = {ops['fs']}  (must stay 10 to match christian's bin_size)")
print(f"  ops['tau']       = {ops['tau']}")
print(f"  ops['diameter']  = {ops['diameter']}")
print(f"  ops['anatomical_only'] = {ops['anatomical_only']}")
print(f"  ops['cellprob_threshold'] = {ops['cellprob_threshold']}")
print(f"  ops['flow_threshold']     = {ops['flow_threshold']}")
print(f"  ops['save_path'] = {ops['save_path']}")
print(f"  wrote {OUT / 'ops.npy'}")

  ops['fs']        = 10.0  (must stay 10 to match christian's bin_size)
  ops['tau']       = 1.3
  ops['diameter']  = 6.076507568359375
  ops['anatomical_only'] = 4
  ops['cellprob_threshold'] = -6
  ops['flow_threshold']     = 0
  ops['save_path'] = D:\jeff\cjennings\zplane02_replication
  wrote D:\jeff\cjennings\zplane02_replication\ops.npy


In [4]:
# Environment audit. If counts don't match, this is the first place to look
# for what changed since christian's run.
import suite2p
import cellpose
from suite2p import pipeline
from suite2p.io.binary import BinaryFile
import inspect

print(f"suite2p pipeline    : {inspect.getsourcefile(pipeline)}")
print(f"cellpose package    : {cellpose.__file__}")
print(f"torch               : {torch.__version__}")
print(f"cuda available      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"cuda device         : {torch.cuda.get_device_name(0)}")
print(f"pipeline signature  : {[p for p in inspect.signature(pipeline).parameters]}")

suite2p pipeline    : C:\Users\RBO\repos\LBM-Suite2p-Python\.venv\Lib\site-packages\suite2p\run_s2p.py
cellpose package    : C:\Users\RBO\repos\LBM-Suite2p-Python\.venv\Lib\site-packages\cellpose\__init__.py
torch               : 2.11.0+cu126
cuda available      : True
cuda device         : NVIDIA RTX A4000
pipeline signature  : ['f_reg', 'f_raw', 'f_reg_chan2', 'f_raw_chan2', 'run_registration', 'ops', 'stat']


In [6]:
# Run upstream suite2p detection-only against christian's data.bin.
# This bypasses the LBM fork's fs-clobber entirely.
Ly, Lx, n_frames = ops["Ly"], ops["Lx"], ops["nframes"]
print(f"opening: {ops['reg_file']}  Ly={Ly} Lx={Lx} n_frames={n_frames}")

with BinaryFile(Ly=Ly, Lx=Lx, filename=ops["reg_file"], n_frames=n_frames) as f_reg:
    ops_out = pipeline(
        f_reg,
        f_raw=None,
        f_reg_chan2=None,
        f_raw_chan2=None,
        run_registration=False,
        ops=ops,
        stat=None,
    )

opening: D:\jeff\cjennings\zplane02_tp00001-08440\data.bin  Ly=1002 Lx=725 n_frames=8440
NOTE: applying default C:\Users\RBO\.suite2p\classifiers\classifier_user.npy
----------- ROI DETECTION
Binning movie in chunks of length 13
Binned movie of size [649,1000,723] created in 13.65 sec.
>>>> CELLPOSE finding masks in max_proj


cellpose.models: model_type argument is not used in v4.0.1+. Ignoring this argument...
cellpose.core: ** TORCH CUDA version installed and working. **
cellpose.core: >>>> using GPU (CUDA)


!NOTE! diameter set to 6.18 for cell detection with cellpose


cellpose.models: >>>> loading model C:\Users\RBO\.cellpose\models\cpsam
cellpose.models: channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used


>>>> 1545 masks detected, median diameter = 6.18 
Detected 1545 ROIs, 17.45 sec
After removing overlaps, 1545 ROIs remain
----------- Total 33.26 sec.
----------- EXTRACTION
Masks created, 1.88 sec.
Extracted fluorescence from 1545 ROIs in 8440 frames, 10.73 sec.
----------- Total 13.16 sec.
----------- CLASSIFICATION
['skew', 'npix_norm', 'compact']
----------- SPIKE DECONVOLUTION
----------- Total 0.39 sec.


In [ ]:
# upstream pipeline writes stat.npy / iscell.npy / F.npy / etc. directly to
# ops['save_path']. Reload them for the comparison cells below.
stat = np.load(OUT / "stat.npy", allow_pickle=True)
iscell = np.load(OUT / "iscell.npy")
print(f"saved outputs in {OUT}:")
for f in sorted(OUT.glob("*.npy")):
    print(f"  {f.name:25s}  {f.stat().st_size/1e6:8.2f} MB")

In [ ]:
# Headline comparison.
n_ours = len(stat)
n_ours_accepted = int(iscell[:, 0].sum())
n_ref = len(ref_stat)
n_ref_accepted = int(ref_iscell[:, 0].sum())

print("=== detection comparison ===")
print(f"  total ROIs    : ours={n_ours:5d}   christian={n_ref:5d}   diff={n_ours - n_ref:+d}")
print(f"  accepted ROIs : ours={n_ours_accepted:5d}   christian={n_ref_accepted:5d}   diff={n_ours_accepted - n_ref_accepted:+d}")

# npix distribution comparison (very telling - tight median + small max means
# cellpose was splitting cells uniformly; sprawling distribution means raw masks)
ours_npix = np.array([s["npix"] for s in stat])
ref_npix = np.array([s["npix"] for s in ref_stat])
print()
print("=== npix distribution (all ROIs) ===")
for label, a in [("ours", ours_npix), ("christian", ref_npix)]:
    print(f"  {label:10s}: min={a.min():4d}  median={np.median(a):6.1f}  mean={a.mean():6.1f}  max={a.max():5d}")

In [ ]:
# Pixel-level comparison of the cellpose-input image. If max_proj matches
# but ROI count differs, the gap is in cellpose itself (model weights, GPU
# nondeterminism, version drift). If max_proj differs, the gap is in binning
# / high-pass / preprocessing.
ours_ops = np.load(OUT / "ops.npy", allow_pickle=True).item()
if "max_proj" in ours_ops and "max_proj" in ref_ops:
    mp_o = np.asarray(ours_ops["max_proj"]).astype(np.float32)
    mp_r = np.asarray(ref_ops["max_proj"]).astype(np.float32)
    if mp_o.shape == mp_r.shape:
        diff = mp_o - mp_r
        corr = np.corrcoef(mp_o.ravel(), mp_r.ravel())[0, 1]
        print("=== max_proj (cellpose input for anatomical_only=4) ===")
        print(f"  ours      : mean={mp_o.mean():.2f}  std={mp_o.std():.2f}  min={mp_o.min():.2f}  max={mp_o.max():.2f}")
        print(f"  christian : mean={mp_r.mean():.2f}  std={mp_r.std():.2f}  min={mp_r.min():.2f}  max={mp_r.max():.2f}")
        print(f"  correlation: {corr:.6f}")
        print(f"  abs diff   : mean={np.abs(diff).mean():.2f}  max={np.abs(diff).max():.2f}")
    else:
        print(f"max_proj shape mismatch: {mp_o.shape} vs {mp_r.shape}")

In [ ]:
# Visual check: side-by-side max_proj.
import matplotlib.pyplot as plt

if "max_proj" in ours_ops and "max_proj" in ref_ops:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    mp_o = np.asarray(ours_ops["max_proj"])
    mp_r = np.asarray(ref_ops["max_proj"])
    vmax = max(mp_o.max(), mp_r.max())
    axes[0].imshow(mp_r, cmap="gray", vmax=vmax); axes[0].set_title(f"christian (n_rois={n_ref})")
    axes[1].imshow(mp_o, cmap="gray", vmax=vmax); axes[1].set_title(f"ours      (n_rois={n_ours})")
    axes[2].imshow(mp_o - mp_r, cmap="RdBu_r", vmin=-vmax/4, vmax=vmax/4); axes[2].set_title("diff (ours - christian)")
    for a in axes: a.axis("off")
    plt.tight_layout()
    plt.savefig(OUT / "max_proj_comparison.png", dpi=120)
    plt.show()
    print(f"saved {OUT / 'max_proj_comparison.png'}")